# ShearNet quickstart

An end-to-end tour of ShearNet in pure Python: **simulate → train → evaluate → visualise**.

Everything here runs on *simulated* data with **no external files**. When no COSMOS
catalog is configured, `generate_dataset` falls back to a synthetic galaxy catalog,
so you can run this notebook top-to-bottom right after `make install`.

**What you'll do**
1. Simulate galaxy postage stamps with known shear `(g1, g2)`.
2. Train a small CNN to recover the shear.
3. Evaluate it and visualise predictions & residuals.

> For real experiments, drive training from the command line with
> `shearnet-train` / `shearnet-eval` (see the README). This notebook uses the
> same building blocks, interactively.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import jax.random as random

from shearnet.core.dataset import generate_dataset
from shearnet.core.train import train_model
from shearnet.metrics import eval_model

## 1 · Configuration

Small, fast defaults so the notebook finishes in a few minutes on CPU. Increase
`N_SAMPLES` / `EPOCHS` for a more accurate model.

In [ ]:
OUTPUT_KEYS = ("g1", "g2")   # parameters the network predicts
N_SAMPLES   = 3000           # total stamps to simulate (split into train/test below)
TRAIN_FRAC  = 0.8            # fraction used for training
EPOCHS      = 5
BATCH_SIZE  = 64
PSF_FWHM    = 0.25           # arcsec — analytic Gaussian PSF ("ideal" mode)
NOISE_SD    = 1e-5
SEED        = 42

rng_key = random.PRNGKey(SEED)

## 2 · Simulate data

`generate_dataset` renders exponential galaxies sheared by catalog values,
convolves them with the PSF, and adds Gaussian noise. Each stamp `i` is
deterministic in `i`, so we simulate **one** dataset and slice it into disjoint
train / test sets (rather than regenerating, which would reuse the same objects).

In [ ]:
images, labels = generate_dataset(
    N_SAMPLES,
    PSF_FWHM,
    nse_sd=NOISE_SD,
    seed=SEED,
    output_keys=OUTPUT_KEYS,
)

split = int(TRAIN_FRAC * N_SAMPLES)
train_images, train_labels = images[:split], labels[:split]
test_images,  test_labels  = images[split:], labels[split:]

print("train:", train_images.shape, train_labels.shape)
print("test: ", test_images.shape,  test_labels.shape)

In [ ]:
# Peek at a few simulated stamps.
n_show = 5
fig, axes = plt.subplots(1, n_show, figsize=(2.2 * n_show, 2.4))
for k in range(n_show):
    axes[k].imshow(train_images[k], cmap="gray")
    axes[k].set_title(f"g1={train_labels[k,0]:+.2f}\ng2={train_labels[k,1]:+.2f}", fontsize=9)
    axes[k].axis("off")
plt.tight_layout()
plt.show()

## 3 · Train a CNN

`train_model` builds the architecture, trains with AdamW + a warmup/cosine
schedule, holds out `val_split` internally for validation, and early-stops on the
best validation loss.

We train on raw shear labels here for simplicity. The `shearnet-train` CLI
additionally z-scores the labels with `shearnet.utils.normalization` (helpful when
predicting parameters of very different magnitudes, e.g. flux vs. g1); pass
`save_path=...` to persist a checkpoint you can reload later.

In [ ]:
state, train_loss, val_loss, val_loss_per_key = train_model(
    train_images,
    train_labels,
    rng_key,
    nn="cnn",                     # 'mlp' | 'cnn' | 'resnet' | 'research_backed' | ...
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    output_keys=OUTPUT_KEYS,
    model_name="quickstart_demo",
    save_path=None,               # set a directory to persist the best checkpoint
)

In [ ]:
plt.figure(figsize=(7, 5))
plt.plot(range(1, len(train_loss) + 1), train_loss, label="train")
plt.plot(range(1, len(val_loss) + 1), val_loss, label="validation")
plt.yscale("log")
plt.xlabel("epoch")
plt.ylabel("MSE loss")
plt.title("Learning curve")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 4 · Evaluate

`eval_model` runs the network over the test set in batches and reports MSE / bias
overall and per output key.

In [ ]:
results = eval_model(state, test_images, test_labels, output_keys=OUTPUT_KEYS)
preds = np.asarray(results["all_preds"])

print(f"Test MSE:  {float(results['loss']):.3e}")
print(f"Test bias: {float(results['bias']):+.3e}")
for key in OUTPUT_KEYS:
    print(f"  {key}: MSE={float(results['loss_per_label'][key]):.3e}  "
          f"bias={float(results['bias_per_label'][key]):+.3e}")

## 5 · Visualise predictions

The plots below adapt to whatever is in `OUTPUT_KEYS`, so they keep working if you
train on more parameters (e.g. `("g1", "g2", "hlr", "flux")`).

In [ ]:
# True vs predicted, one panel per output key.
fig, axes = plt.subplots(1, len(OUTPUT_KEYS), figsize=(5 * len(OUTPUT_KEYS), 5))
axes = np.atleast_1d(axes)
for ax, key in zip(axes, OUTPUT_KEYS):
    j = OUTPUT_KEYS.index(key)
    t, p = test_labels[:, j], preds[:, j]
    lo, hi = np.percentile(t, [1, 99])
    ax.scatter(t, p, s=8, alpha=0.4)
    ax.plot([lo, hi], [lo, hi], "r--", label="y = x")
    rmse = float(np.sqrt(np.mean((p - t) ** 2)))
    bias = float(np.mean(p - t))
    ax.set_title(f"{key}   RMSE={rmse:.2e}, bias={bias:+.2e}")
    ax.set_xlabel(f"{key} true")
    ax.set_ylabel(f"{key} predicted")
    ax.set_aspect("equal", adjustable="box")
    ax.legend()
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Residual distributions (pred - true).
fig, axes = plt.subplots(1, len(OUTPUT_KEYS), figsize=(5 * len(OUTPUT_KEYS), 4))
axes = np.atleast_1d(axes)
for ax, key in zip(axes, OUTPUT_KEYS):
    j = OUTPUT_KEYS.index(key)
    res = preds[:, j] - test_labels[:, j]
    ax.hist(res, bins=50, alpha=0.8)
    ax.axvline(0, color="red", ls="--")
    ax.axvline(float(np.mean(res)), color="k", ls="-", lw=1, label=f"mean={np.mean(res):+.2e}")
    ax.set_title(f"{key} residuals")
    ax.set_xlabel("pred − true")
    ax.set_ylabel("count")
    ax.legend()
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Next steps

- **Persist a model** — pass `save_path="..."` to `train_model`, or use
  `shearnet-train --config configs/example.yaml`.
- **Compare several trained models** — `02_model_comparison.ipynb`.
- **Build input catalogs** from COSMOS / detection data — `03_catalog_builder.ipynb`.
- **Inspect PSFs and PSF leakage** — `04_psf_diagnostics.ipynb`.
- **Add an NGmix metacalibration baseline** — `eval_ngmix` in `shearnet.metrics`
  (needs `return_obs=True` from `generate_dataset`).